# Determining the Optimal Number of Hidden Layers and Neurons for an ANN

1. **Begin with simple architecture and gradually increase complexity if needed.**
   
2. **Grid Search/Random Search**: Use to try different architectures.
   
3. **Cross Validation**: Use to evaluate performance of different architectures.
   
4. **Heuristics and Rules of Thumb**: 
    - The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
    - A common practice is to start with 1-2 hidden layers.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Input

In [7]:
from scikeras.wrappers import KerasRegressor

In [2]:
data = pd.read_csv("Churn_Modelling.csv")

data.drop(["RowNumber", "CustomerId", "Surname"] , axis=1, inplace=True)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()

geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)


X = data.drop("EstimatedSalary", axis=1)
y = data["EstimatedSalary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train  = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

#### Define a function to create a model and try different parameters (KerasClassifer)

In [10]:
def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(neurons, activation='relu'))

    for _ in range(layers-1): # Hidden layers only
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

    return model

#### Create Keras Classifier

In [11]:
model = KerasRegressor(neurons=32, layers=1, build_fn=create_model, verbose=1)

#### Grid Search Parameters

In [12]:
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1,2,3],
    'epochs': [50, 100]
}

#### Perform Gid Search

In [13]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


C:\Users\gurunaml\OneDrive - Firstsource Solutions Ltd\Desktop\ML\ML\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 99679.6328 - mae: 99679.6328   
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 99501.6953 - mae: 99501.6953    
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 88716.5625 - mae: 88716.5625  
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 69580.8750 - mae: 69580.8750  
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 53620.1289 - mae: 53620.1289  
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 51131.4297 - mae: 51131.4297  
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 50410.0039 - mae: 50410.0039  
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 50568.5039 - mae: 50568.5039  
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49920.2578 - mae: 49920.2578  
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49746.2070 - mae: 49746.2070  
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49822.9688 - m

In [14]:
grid_result.best_params_

{'epochs': 100, 'layers': 3, 'neurons': 16}

In [15]:
grid_result.best_score_

-0.015214593615531532

In [16]:
grid_result.best_estimator_

KerasRegressor(
	model=None
	build_fn=<function create_model at 0x000002023270C9A0>
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=None
	validation_batch_size=None
	verbose=1
	callbacks=None
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=100
	neurons=16
	layers=3
)